<a href="https://colab.research.google.com/github/PeesapatiRohithSharma/Summer_2026/blob/main/decision_trees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import numpy as np
import pandas as pd

In [7]:
class Node:
  def __init__(self, gini, majority_class,num_samples,num_samples_per_class, num_classes,threshold=None,feature_index=None):
    self.gini =  gini
    self.num_samples = num_samples
    self.per_class = num_samples_per_class
    self.threshold = threshold
    self.feature_index = feature_index
    self.majority_class = majority_class
    self.left = None
    self.right = None
    self.num_classes = num_classes

In [8]:
class DecisionTree:
  def __init__(self, max_depth, min_samples):
    self.max_depth=max_depth
    self.min_samples = min_samples
    self.root = None

  def gini_impurity(self, y):
    return 1.0 - sum((np.sum(y==c)/len(y))**2 for c in range(self.num_classes))

  def best_split(self, X, y):
    best_gini = 1
    best_feature = None
    best_threshold = None
    for feature in range(X.shape[1]):
      X_f = X[:,feature]
      for t in np.unique(X_f):
        left_split = X_f < t
        right_split = ~left_split
        y_left = y[left_split]
        y_right = y[right_split]
        if len(y_left) == 0 or len(y_right) == 0:
          continue
        gini = (len(y_left)/len(y))*(self.gini_impurity(y_left)) + (len(y_right)/len(y))*(self.gini_impurity(y_right))
        if gini < best_gini:
          best_gini = gini
          best_feature = feature
          best_threshold = t
    return best_feature, best_threshold

  def grow_tree(self,X,y,depth):
    n_samples = len(y)
    n_classes = len(np.unique(y))
    arr, counts = np.unique(y, return_counts=True)
    majority_class = arr[np.argmax(counts)]
    f,t = self.best_split(X,y)
    node = Node(self.gini_impurity(y),majority_class,n_samples,counts,n_classes,f,t)
    if f is None or t is None:
      return node
    if depth >= self.max_depth or len(np.unique(y)) == 1 or len(y) < self.min_samples:
      return node

    else:
      left_indices = X[:, f] < t
      right_indices = ~left_indices
      X_left, y_left = X[left_indices], y[left_indices]
      X_right, y_right = X[right_indices], y[right_indices]
      node.feature_index = f
      node.threshold = t
      node.left = self.grow_tree(X_left, y_left, depth+1)
      node.right = self.grow_tree(X_right, y_right, depth+1)
    return node

  def predict_class(self,input):
    node = self.root
    while node.feature_index is not None:
      if input[node.feature_index] < node.threshold:
        node = node.left
      else:
        node = node.right

    return node.majority_class

  def fit(self,X,y):
    self.num_classes = len(np.unique(y))
    self.root = self.grow_tree(X,y,0)